# Tutorial 6-2: The Magic of Random Projections
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

--- 
### Verifying the Johnson-Lindenstrauss Lemma


## 1. The Theory: Why Randomness Works
In high-dimensional spaces, a surprising geometric property emerges: most vectors are nearly orthogonal to each other. The **Johnson-Lindenstrauss (JL) Lemma** provides a mathematical guarantee that a set of points in a high-dimensional space can be embedded into a much lower-dimensional space $k$ while keeping pairwise distances nearly intact.

The lemma states that for any $0 < \epsilon < 1$, there exists a mapping $A: \mathbb{R}^{d} \rightarrow \mathbb{R}^{k}$ such that for all points $x$ and $z$:
$$(1-\epsilon)||x-z||^{2} \le \frac{d}{k}||xA-zA||^{2} \le (1+\epsilon)||x-z||^{2}$$

Surprisingly, this mapping $A$ can be a **random matrix**. We don't need to learn the structure of the data to preserve its geometry; we just need enough dimensions ($k$) to satisfy the logarithmic requirement of the lemma.

## 2. Gaussian vs. Sparse (Achlioptas) Projections
We will compare two ways to define our random matrix $A$:
1. **Gaussian Random Matrix:** Each element $A(i,j)$ is drawn from a normal distribution $N(0, 1)$.
2. **Sparse (Achlioptas) Matrix:** A more computationally efficient version where elements are drawn from {1, 0, -1} with probabilities {1/6, 2/3, 1/6}. This is "database-friendly" because it replaces most floating-point multiplications with simple additions or zeros.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.random_projection import GaussianRandomProjection, SparseRandomProjection
from sklearn.metrics.pairwise import euclidean_distances

# Step 1: Generate a high-dimensional synthetic dataset
# 500 samples in a 10,000-dimensional space
n_samples = 500
d_features = 10000
np.random.seed(42)
X = np.random.randn(n_samples, d_features)

# Calculate the "Ground Truth" pairwise distances in the original 10,000D space
true_dists = euclidean_distances(X)
# We only need the unique pairwise distances (upper triangle of the matrix)
true_dists_flat = true_dists[np.triu_indices(n_samples, k=1)]

## 3. Performing the Projections
We will now reduce the data from **10,000 dimensions down to 500 dimensions** (a 20x reduction) using both random methods.

In [ ]:
k = 500

# Gaussian Random Projection
grp = GaussianRandomProjection(n_components=k, random_state=42)
X_grp = grp.fit_transform(X)
grp_dists = euclidean_distances(X_grp)[np.triu_indices(n_samples, k=1)]

# Sparse (Achlioptas) Random Projection
srp = SparseRandomProjection(n_components=k, random_state=42)
X_srp = srp.fit_transform(X)
srp_dists = euclidean_distances(X_srp)[np.triu_indices(n_samples, k=1)]

## 4. Measuring Distance Preservation
If the projection is faithful, the ratio of the projected distance to the original distance should be approximately 1.0. We will plot the distribution of these ratios to see how much "distortion" or noise was introduced by the reduction.

In [ ]:
plt.figure(figsize=(12, 6))

plt.hist(grp_dists / true_dists_flat, bins=100, alpha=0.5, label='Gaussian Projection', color='blue')
plt.hist(srp_dists / true_dists_flat, bins=100, alpha=0.5, label='Sparse (Achlioptas) Projection', color='green')

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Preservation')
plt.title(f"Pairwise Distance Ratios (10,000D reduced to {k}D)")
plt.xlabel("Ratio (Projected Distance / Original Distance)")
plt.ylabel("Count of Pairs")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## 5. Analyzing the Trade-off: k vs. Error
The JL Lemma suggests that the error $\epsilon$ depends on $k$. As we increase the number of target dimensions, the preservation should become more accurate (tighter distribution around 1.0).

In [ ]:
k_values = [50, 200, 500, 1000]
for val in k_values:
    proj = GaussianRandomProjection(n_components=val, random_state=42)
    X_proj = proj.fit_transform(X)
    p_dists = euclidean_distances(X_proj)[np.triu_indices(n_samples, k=1)]
    
    # Calculate the standard deviation of the distance ratio
    # A smaller std dev means higher fidelity to the original geometry
    std_error = np.std(p_dists / true_dists_flat)
    print(f"Target Dimensions (k): {val:4d} | Distortion (Std Dev): {std_error:.4f}")

## Summary
1. **Speed:** Unlike PCA or SVD, Random Projections do not require expensive matrix decompositions, making them ideal for high-dimensional data.
2. **Fidelity:** Even with a significant reduction in dimensions (e.g., from 10,000 to 500), the pairwise distances are preserved with remarkably low error.
3. **Sparsity:** The Achlioptas method provides nearly identical results to the Gaussian method while being significantly faster to compute, as it avoids most floating-point operations.